In [4]:
"""Summarize point-based daily IMERG rainfall from downloaded files."""
# https://search.earthdata.nasa.gov/search/granules?p=C2723754864-GES_DISC!C2723754864-GES_DISC&pg[1][v]=t&pg[1][qt]=2024-01-01T00%3A00%3A00.000Z%2C2025-01-01T23%3A59%3A59.999Z&pg[1][gsk]=-start_date&pg[1][m]=download&pg[1][cd]=f&sb[0]=88.48304%2C22.44988%2C91.44029%2C25.1027&lat=23.8514206460103&long=91.20536672079196&zoom=6.669064366439985
# Requires: h5netcdf, numpy, pandas, scipy, xarray

from __future__ import annotations

import calendar
import re
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
import xarray as xr


LAT = 23.8103
LON = 90.4125
START_YEAR = 2000
END_YEAR = 2025
THRESHOLD_BASELINE_START = 2001
THRESHOLD_BASELINE_END = 2025
ONSET_START_DAY_OF_YEAR = 121
ONSET_END_DAY_OF_YEAR = 270
GRID_LON_FIRST = -179.95
GRID_LAT_FIRST = -89.95
GRID_STEP = 0.1
GRID_LON_COUNT = 3600
GRID_LAT_COUNT = 1800
PRECIPITATION_VARIABLES = ("precipitation", "precipitationCal")
WET_DAY_THRESHOLD_MM = 1.0
DRY_STRETCH_THRESHOLD_MM = 5.0
FIVE_DAY_WINDOW = 5
DRY_STRETCH_WINDOW = 10
DRY_STRETCH_CHECK_DAYS = 30

BASE_DIR = Path.cwd()
SEARCH_ROOTS = {BASE_DIR, *BASE_DIR.parents}
DATA_DIRS = sorted(
    {
        path.resolve()
        for root in SEARCH_ROOTS
        for folder in (
            root,
            root / "imerg_monthly",
            root / "data" / "datacollect" / "nasa" / "rain_imperg" / "imerg_monthly",
        )
        for path in folder.glob("GPM_3IMERGDF_07-*")
        if path.is_dir()
    }
)
if len(DATA_DIRS) != 1:
    raise FileNotFoundError(
        f"Expected one GPM_3IMERGDF_07-* folder inside {BASE_DIR}, "
        f"found {len(DATA_DIRS)}. Run the notebook from the folder containing the GPM data."
    )
DATA_DIR = DATA_DIRS[0]
DATASETS_DIRS = sorted(
    {
        (root / "data" / "datasets").resolve()
        for root in SEARCH_ROOTS
        if (root / "data" / "datasets").is_dir()
    }
)
if len(DATASETS_DIRS) != 1:
    raise FileNotFoundError(
        f"Expected one data\\datasets folder near {BASE_DIR}, "
        f"found {len(DATASETS_DIRS)}."
    )
OUTPUT_DIR = DATASETS_DIRS[0]
DAILY_CSV = OUTPUT_DIR / "imerg_daily_point.csv"
SEASON_CSV = OUTPUT_DIR / "season.csv"
ONSET_CSV = OUTPUT_DIR / "monsoon_onset_by_year.csv"
COMPARISON_CSV = OUTPUT_DIR / "onset_comparison.csv"


def nearest_grid_index(
    coordinate: float,
    first_cell_center: float,
    spacing: float,
    count: int,
) -> tuple[int, float]:
    index = int(round((coordinate - first_cell_center) / spacing))
    if not 0 <= index < count:
        raise ValueError(f"Coordinate {coordinate} is outside the IMERG grid.")
    return index, first_cell_center + index * spacing


def read_downloaded_daily_data() -> tuple[pd.DataFrame, float, float]:
    longitude_index, grid_lon = nearest_grid_index(
        LON, GRID_LON_FIRST, GRID_STEP, GRID_LON_COUNT
    )
    latitude_index, grid_lat = nearest_grid_index(
        LAT, GRID_LAT_FIRST, GRID_STEP, GRID_LAT_COUNT
    )

    dated_files: dict[date, Path] = {}
    for path in sorted(DATA_DIR.glob("*.nc4")):
        match = re.search(r"\.3IMERG\.(\d{8})-", path.name)
        if match is None:
            raise ValueError(f"Could not read a daily date from {path.name!r}.")
        day = date.fromisoformat(
            f"{match.group(1)[:4]}-{match.group(1)[4:6]}-{match.group(1)[6:]}"
        )
        if day in dated_files:
            raise ValueError(f"More than one downloaded IMERG file exists for {day}.")
        dated_files[day] = path

    if not dated_files:
        raise FileNotFoundError(f"No daily .nc4 files found in {DATA_DIR}.")

    rows: list[dict[str, object]] = []
    unreadable_files: list[Path] = []
    for day, path in sorted(dated_files.items()):
        try:
            with xr.open_dataset(path, engine="h5netcdf") as dataset:
                variable = next(
                    (name for name in PRECIPITATION_VARIABLES if name in dataset.data_vars),
                    None,
                )
                if variable is None:
                    raise KeyError(
                        f"No supported precipitation variable in {path.name}: "
                        f"{list(dataset.data_vars)}"
                    )
                values = dataset[variable].isel(
                    lon=longitude_index,
                    lat=latitude_index,
                ).values
                if values.size != 1:
                    raise ValueError(
                        f"Expected one daily point value in {path.name}, "
                        f"found {values.size}."
                    )
                rainfall = float(np.asarray(values).reshape(-1)[0])
                if rainfall <= -9000:
                    rainfall = np.nan
        except OSError as error:
            unreadable_files.append(path)
            print(f"WARNING: skipped unreadable file {path.name}: {error}")
            continue
        rows.append({"date": day, "precipitation_mm_day": rainfall})

    if not rows:
        raise RuntimeError(f"No readable daily IMERG files found in {DATA_DIR}.")

    observed = pd.DataFrame(rows).set_index("date").sort_index()
    first_date = observed.index.min()
    last_date = observed.index.max()
    global START_YEAR, END_YEAR, THRESHOLD_BASELINE_START, THRESHOLD_BASELINE_END
    START_YEAR = first_date.year
    END_YEAR = last_date.year
    THRESHOLD_BASELINE_START = max(2001, START_YEAR)
    THRESHOLD_BASELINE_END = END_YEAR
    dates = pd.date_range(
        first_date,
        last_date,
        freq="D",
        name="date",
    )
    daily = pd.DataFrame(index=dates)
    daily["precipitation_mm_day"] = observed["precipitation_mm_day"].reindex(dates)
    daily["latitude"] = grid_lat
    daily["longitude"] = grid_lon
    daily.reset_index().to_csv(DAILY_CSV, index=False)
    print(f"Read {len(rows)} of {len(dated_files)} local daily files from {DATA_DIR}")
    if unreadable_files:
        print(f"Re-download these {len(unreadable_files)} incomplete or unreadable files.")

    return daily, grid_lat, grid_lon


def make_monthly_summary(daily: pd.DataFrame) -> pd.DataFrame:
    data = daily.copy()
    data.index = pd.to_datetime(data.index)
    data["year"] = data.index.year
    data["month"] = data.index.month
    data["wet_day"] = data["precipitation_mm_day"] >= WET_DAY_THRESHOLD_MM

    monthly = data.groupby(["year", "month"]).agg(
        valid_days=("precipitation_mm_day", "count"),
        observed_month_total_mm=("precipitation_mm_day", "sum"),
        rainy_days=("wet_day", "sum"),
    )
    monthly["days_in_month"] = [
        calendar.monthrange(year, month)[1] for year, month in monthly.index
    ]
    monthly["complete_month"] = monthly["valid_days"] == monthly["days_in_month"]
    monthly["monthly_total_mm"] = monthly["observed_month_total_mm"].where(
        monthly["complete_month"]
    )
    monthly = monthly.reset_index()

    complete_months = monthly[monthly["complete_month"]]
    rainfall_by_month = data.groupby("month")["precipitation_mm_day"]
    monthly_rainfall_mean = rainfall_by_month.mean()
    wet_day_frequency = (
        data.groupby("month")["wet_day"].sum()
        / data.groupby("month")["precipitation_mm_day"].count()
        * 100
    )
    summary = complete_months.groupby("month").agg(
        years_with_complete_data=("year", "nunique"),
        mean_monthly_total_mm=("monthly_total_mm", "mean"),
        median_monthly_total_mm=("monthly_total_mm", "median"),
        mean_rainy_days=("rainy_days", "mean"),
    )
    summary["mean_daily_precip_mm_day"] = monthly_rainfall_mean
    summary["wet_day_frequency_percent"] = wet_day_frequency
    summary["wet_day_threshold_mm"] = WET_DAY_THRESHOLD_MM
    summary["first_year"] = START_YEAR
    summary["last_year"] = END_YEAR
    summary = summary.reindex(range(1, 13))
    summary.index.name = "month"
    summary = summary.reset_index()
    summary["month_name"] = summary["month"].map(
        lambda month: calendar.month_name[month]
    )
    return summary[
        [
            "month",
            "month_name",
            "years_with_complete_data",
            "mean_monthly_total_mm",
            "median_monthly_total_mm",
            "mean_daily_precip_mm_day",
            "mean_rainy_days",
            "wet_day_frequency_percent",
            "wet_day_threshold_mm",
            "first_year",
            "last_year",
        ]
    ]


def rolling_five_day_totals(
    rainfall: pd.Series,
    years: range,
) -> dict[date, float]:
    totals: dict[date, float] = {}
    for year in years:
        for day_of_year in range(
            ONSET_START_DAY_OF_YEAR,
            ONSET_END_DAY_OF_YEAR + 1,
        ):
            start = date(year, 1, 1) + timedelta(days=day_of_year - 1)
            window = rainfall.reindex(
                pd.date_range(start, start + timedelta(days=FIVE_DAY_WINDOW - 1))
            )
            if len(window) == FIVE_DAY_WINDOW and window.notna().all():
                totals[start] = float(window.sum())
    return totals


def detect_monsoon_onsets(
    daily: pd.DataFrame,
) -> tuple[pd.DataFrame, float]:
    rainfall = daily["precipitation_mm_day"].copy()
    rainfall.index = pd.to_datetime(rainfall.index).normalize()
    baseline_totals = rolling_five_day_totals(
        rainfall,
        range(THRESHOLD_BASELINE_START, THRESHOLD_BASELINE_END + 1),
    )
    if not baseline_totals:
        raise ValueError("No complete five-day rainfall windows are available for 2001–2025.")
    threshold = float(np.median(list(baseline_totals.values())))

    onset_rows: list[dict[str, object]] = []
    for year in range(START_YEAR, END_YEAR + 1):
        if year == 2000:
            onset_rows.append(
                {
                    "year": year,
                    "onset_date": pd.NA,
                    "onset_day_after_may_1": np.nan,
                    "status": "incomplete_year_coverage",
                    "five_day_threshold_mm": threshold,
                }
            )
            continue

        incomplete_candidate = False
        onset_date = None
        for start, five_day_total in rolling_five_day_totals(rainfall, range(year, year + 1)).items():
            if five_day_total < threshold:
                continue
            following = rainfall.reindex(
                pd.date_range(
                    start + timedelta(days=1),
                    start + timedelta(days=DRY_STRETCH_CHECK_DAYS),
                )
            )
            if len(following) != DRY_STRETCH_CHECK_DAYS or following.isna().any():
                incomplete_candidate = True
                continue
            dry_stretch = (
                following.rolling(DRY_STRETCH_WINDOW, min_periods=DRY_STRETCH_WINDOW)
                .sum()
                .lt(DRY_STRETCH_THRESHOLD_MM)
                .any()
            )
            if not dry_stretch:
                onset_date = start
                break

        status = (
            "found"
            if onset_date is not None
            else "incomplete_data"
            if incomplete_candidate
            else "no_onset_found"
        )
        onset_rows.append(
            {
                "year": year,
                "onset_date": onset_date.isoformat() if onset_date else pd.NA,
                "onset_day_after_may_1": (
                    (onset_date - date(year, 5, 1)).days
                    if onset_date
                    else np.nan
                ),
                "status": status,
                "five_day_threshold_mm": threshold,
            }
        )
    return pd.DataFrame(onset_rows), threshold


def make_onset_comparison(onsets: pd.DataFrame, threshold: float) -> pd.DataFrame:
    found = onsets[onsets["status"] == "found"].copy()
    early = found.loc[
        found["year"].between(2001, 2012), "onset_day_after_may_1"
    ].astype(float)
    late = found.loc[
        found["year"].between(2013, 2025), "onset_day_after_may_1"
    ].astype(float)

    if early.empty or late.empty:
        statistic = np.nan
        p_value = np.nan
        conclusion = "insufficient valid onset years for both periods"
    else:
        result = mannwhitneyu(early, late, alternative="two-sided", method="auto")
        statistic = float(result.statistic)
        p_value = float(result.pvalue)
        conclusion = (
            "onset timing differs at alpha=0.05"
            if p_value < 0.05
            else "no statistically significant onset shift at alpha=0.05"
        )

    return pd.DataFrame(
        [
            {
                "early_period": "2001-2012",
                "early_valid_onset_years": len(early),
                "early_median_days_after_may_1": (
                    float(early.median()) if not early.empty else np.nan
                ),
                "late_period": "2013-2025",
                "late_valid_onset_years": len(late),
                "late_median_days_after_may_1": (
                    float(late.median()) if not late.empty else np.nan
                ),
                "test": "two-sided Mann-Whitney U",
                "mann_whitney_u": statistic,
                "p_value": p_value,
                "alpha": 0.05,
                "five_day_threshold_mm": threshold,
                "conclusion": conclusion,
            }
        ]
    )


def main() -> None:
    daily, grid_lat, grid_lon = read_downloaded_daily_data()
    season = make_monthly_summary(daily)
    onsets, threshold = detect_monsoon_onsets(daily)
    comparison = make_onset_comparison(onsets, threshold)

    season.to_csv(SEASON_CSV, index=False)
    onsets.to_csv(ONSET_CSV, index=False)
    comparison.to_csv(COMPARISON_CSV, index=False)

    available = season.dropna(subset=["mean_monthly_total_mm"])
    if available.empty:
        raise RuntimeError("No complete months are available to summarize.")
    wettest_month = available.loc[
        available["mean_monthly_total_mm"].idxmax(), "month_name"
    ]
    most_frequent_rain_month = available.loc[
        available["wet_day_frequency_percent"].idxmax(), "month_name"
    ]

    print(f"Nearest IMERG grid cell: {grid_lat:.2f} N, {grid_lon:.2f} E")
    print(f"Wettest month in downloaded data by mean total: {wettest_month}")
    print(f"Most frequent wet days in downloaded data: {most_frequent_rain_month}")
    print(f"Downloaded-data five-day threshold (median): {threshold:.2f} mm")
    print(comparison.to_string(index=False))
    print(f"Daily point data: {DAILY_CSV}")
    print(f"Monthly rainfall summary: {SEASON_CSV}")
    print(f"Yearly onset dates: {ONSET_CSV}")
    print(f"Period comparison: {COMPARISON_CSV}")


if __name__ == "__main__":
    main()


Read 367 of 367 local daily files from E:\Users\ismail\Desktop\collector_sheba\data\datacollect\nasa\rain_imperg\imerg_monthly\GPM_3IMERGDF_07-20261105_191606
Nearest IMERG grid cell: 23.85 N, 90.45 E
Wettest month in downloaded data by mean total: August
Most frequent wet days in downloaded data: August
Downloaded-data five-day threshold (median): 56.12 mm
early_period  early_valid_onset_years  early_median_days_after_may_1 late_period  late_valid_onset_years  late_median_days_after_may_1                     test  mann_whitney_u  p_value  alpha  five_day_threshold_mm                                      conclusion
   2001-2012                        0                            NaN   2013-2025                       1                          -1.0 two-sided Mann-Whitney U             NaN      NaN   0.05              56.120002 insufficient valid onset years for both periods
Daily point data: E:\Users\ismail\Desktop\collector_sheba\data\datacollect\nasa\rain_imperg\imerg_monthly\imerg_da

hi
